<a href="https://colab.research.google.com/github/AlexZhuo29/QCC_STEP_Research_2025_2026/blob/main/STEP_Research_Graph_2025_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## import needed libraries
from google.colab import auth
import gspread
from google.auth import default
import pandas as pd
import plotly.graph_objects as go


## imports the results and prepare the data for graphing
def importDF_func(spreadSheet):
  ## authenticate user
  auth.authenticate_user()

  ## create the gspread client
  creds, _ = default()
  gc = gspread.authorize(creds)

  ## checks if the input is a string, if not the function will terminate
  if (type(spreadSheet) != str):
      print('The input is not a string')
      return

  ## opens the spreadsheet provided
  sheet_url = spreadSheet
  spreadsheet = gc.open_by_url(sheet_url)

  ## select the spreadsheet tab
  worksheet = spreadsheet.get_worksheet(0)

  ## turn spreadsheet into dataframe
  rows = worksheet.get_all_values()
  df = pd.DataFrame(rows[1:], columns=rows[0])

  ## removes the column for Train Score
  df = df.drop(columns='Train Score')

  ## converts the Test Score values from string to float
  df['Test Score'] = df['Test Score'].astype(float)

  ## return the dataframe
  return df

## graph the data in the dataframe
def dfToGraph_func(company,dataframe):
  ## Y coordinates of the independent variables / hyper parameters
  values_weights = [None, 'uniform', 'distance']
  values_metrics = ['cosine', 'euclidean', 'manhattan']

  ## encode each value to a number
  values_weights_map = {j: i for i, j in enumerate(values_weights)}
  values_metrics_map = {j: i for i, j in enumerate(values_metrics)}

  ## create the graph
  fig = go.Figure(
      data=go.Parcoords(
          ## this section will setup the color encoding
          line=dict(
              ## this line decides which column to be used for color encoding
              color=dataframe["Test Score"],
              ## visual settings
              colorscale="Viridis",
              showscale=True,
              colorbar=dict(title="Test Score")
          ),
          ## setup for each vertical axises
          dimensions=[
              ## num_neighbors
              dict(
                  ## name of vertical axis
                  label='Num_Neighbors',
                  ## values to graph
                  values=dataframe['N_neighbors'],
                  ## range of values
                  range=[1, 100],
                  ## scale used on the vertical axis
                  tickvals=[1, 10, 19, 28, 37, 46, 55, 64, 73, 82, 91, 100]
              ),
              ## weights
              dict(
                  ## name of vertical axis
                  label="Weights",
                  ## values to graph
                  values=dataframe["Weights"].map(values_weights_map),
                  ## range of values
                  range=[0, len(values_weights) - 1],
                  ## scale used on the vertical axis
                  tickvals=list(range(len(values_weights))),
                  ## label the mapped numbers with the original name
                  ticktext=values_weights
              ),
              ## metrics
              dict(
                  ## name of vertical axis
                  label="Metrics",
                  ## values to graph
                  values=dataframe["Metric"].map(values_metrics_map),
                  ## range of values
                  range=[0, len(values_metrics) - 1],
                  ## scale used on the vertical axis
                  tickvals=list(range(len(values_metrics))),
                  ## label the mapped numbers with the original name
                  ticktext=values_metrics
              ),
              ## power
              dict(
                  ## name of vertical axis
                  label="Power",
                  ## values to graph
                  values=dataframe["Power"],
                  ## range of values
                  range=[1, 100],
                  ## scale used on the vertical axis
                  tickvals=[1, 10, 19, 28, 37, 46, 55, 64, 73, 82, 91, 100]
              ),
          ],
      )
  )

  ## create the title of the graph
  fig.update_layout(title=f"Parallel Coordinates Plot for {company} Stocks")

  ## set the dimensions of the graph
  fig.update_layout(width=2000, height=1000)

  ## display the graph
  fig.show()

def spreadsheetToGraph_func(nameOfCompany,URL):
  dfToGraph_func(nameOfCompany,importDF_func(URL))